In [5]:
import os

In [6]:
%pwd

'C:\\Users\\Riyaz\\OneDrive\\Desktop\\text_summarizer'

In [7]:
import os

os.chdir(r"C:\Users\Riyaz\OneDrive\Desktop\text_summarizer")

print(os.getcwd())

C:\Users\Riyaz\OneDrive\Desktop\text_summarizer


In [8]:
## this is entity class
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class  DataValidationConfig:
    root_dir :Path
    data_path : Path
    tokenizer_name :Path

In [9]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories
from textSummarizer.entity import DataTransformationConfig

In [10]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath= PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])



    def get_data_transformation_config(self) -> DataTransformationConfig:
            config =self.config.data_transformation

            create_directories([config.root_dir])

            data_transformation_config =DataTransformationConfig(
                 root_dir= config.root_dir,
                 data_path=config.data_path,
                 tokenizer_name=config.tokenizer_name,
            )

            return data_transformation_config

In [11]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

c:\Users\Riyaz\OneDrive\Desktop\text_summarizer\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class DataTransformation:
    
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(
            config.tokenizer_name
        )

    def convert_examples_to_features(self, example_batch):

        input_encodings = self.tokenizer(
            example_batch['dialogue'],
            max_length=1024,
            truncation=True
        )

        target_encodings = self.tokenizer(
            example_batch['summary'],
            max_length=128,
            truncation=True
        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):

        dataset_samsum = load_from_disk(
            self.config.data_path
        )

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        dataset_samsum_pt.save_to_disk(
            os.path.join(
                self.config.root_dir,
                "samsum_dataset"
            )
        )

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")
print("Tokenizer loaded successfully")

[2026-05-11 21:32:09,995: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-05-11 21:32:10,405: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-11 21:32:10,712: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 21:32:11,019: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-05-11 21:32:11,326: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 21:32:11,633: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main?recursive=true&e

In [14]:
try:
    config = ConfigurationManager()

    data_transformation_config = config.get_data_transformation_config()

    data_transformation = DataTransformation(
        config=data_transformation_config
    )

    data_transformation.convert()

except Exception as e:
    raise e

[2026-05-11 21:32:11,973: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 21:32:11,978: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 21:32:11,981: INFO: common: created directory at: artifacts]
[2026-05-11 21:32:11,984: INFO: common: created directory at: artifacts/data_transformation]
[2026-05-11 21:32:12,351: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]


[2026-05-11 21:32:12,354: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-05-11 21:32:12,926: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-11 21:32:13,270: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 21:32:13,579: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-05-11 21:32:13,885: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 21:32:14,193: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 70579.51 examples/s]


In [15]:
import os

os.chdir(r"C:\Users\Riyaz\OneDrive\Desktop\text_summarizer")

print(os.getcwd())

C:\Users\Riyaz\OneDrive\Desktop\text_summarizer


In [16]:
config = ConfigurationManager()

data_transformation_config = config.get_data_transformation_config()

data_transformation = DataTransformation(
    config=data_transformation_config
)

data_transformation.convert()

[2026-05-11 21:32:19,490: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-11 21:32:19,494: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-11 21:32:19,499: INFO: common: created directory at: artifacts]
[2026-05-11 21:32:19,501: INFO: common: created directory at: artifacts/data_transformation]


[2026-05-11 21:32:19,927: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-05-11 21:32:20,234: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-11 21:32:20,541: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 21:32:20,850: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[2026-05-11 21:32:21,156: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 21:32:21,486: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main?recursive=true&e

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 77065.15 examples/s]
